# Lab Assignment 3 — Task 2

## Converting a CFG to Chomsky Normal Form (CNF)

Many parsing algorithms, including CKY, require the grammar to be in **Chomsky Normal Form (CNF)**.

A grammar is in CNF if its production rules have one of these forms:

1. `A -> B C` — a non-terminal produces two non-terminals.
2. `A -> 'word'` — a non-terminal produces a single terminal.

## PART 1 : The Challenge of Ambiguity & CNF Constraints

Consider:

> **I saw the man with a telescope.**

This sentence has two possible interpretations:

1. I saw a man who was holding a telescope.
2. I used a telescope to see the man.

A PCFG assigns probabilities to grammar rules. The probability of a complete parse tree is obtained by multiplying the probabilities of all production rules used in that tree.

### Your task

1. **Convert the given PCFG into strict CNF.** Identify the rule that violates CNF and replace it with equivalent binary rule(s). Preserve the probability of the original alternative.
2. **Parse the sentence using the CNF PCFG and `ViterbiParser`.** Print the most probable parse tree and its total probability.
3. **Explain mathematically why the parser selected this interpretation over the alternative.** Show the rule probabilities used in the selected tree and compare the resulting probability with the competing interpretation.

In [ ]:
pcfg_grammar_str = """
    S -> NP VP [1.0]
    NP -> 'I' [0.1] | Det N [0.3] | NP PP [0.6]
    VP -> V NP [0.7] | V NP PP [0.3]
    PP -> P NP [1.0]
    Det -> 'the' [0.8] | 'a' [0.2]
    N -> 'man' [0.5] | 'telescope' [0.5]
    V -> 'saw' [1.0]
    P -> 'with' [1.0]
"""

sentence = "I saw the man with a telescope"

In [ ]:
import nltk
from nltk import PCFG
from nltk.parse import ViterbiParser

# Convert the non-CNF rule:
#     VP -> V NP PP [0.3]
# into binary rules. Choose an intermediate non-terminal and
# distribute the probability so that the original derivation
# keeps the same probability.

# Example:
#     VP -> V VP_PP [0.3]
#     VP_PP -> NP PP [1.0]

cnf_grammar_str = """
    S -> NP VP [1.0]
    NP -> 'I' [0.1] | Det N [0.3] | NP PP [0.6]
    VP -> V NP [0.7] | V VP_PP [0.3]
    VP_PP -> NP PP [1.0]
    PP -> P NP [1.0]
    Det -> 'the' [0.8] | 'a' [0.2]
    N -> 'man' [0.5] | 'telescope' [0.5]
    V -> 'saw' [1.0]
    P -> 'with' [1.0]
"""

In [ ]:
grammar = PCFG.fromstring(cnf_grammar_str)
words = sentence.split()

# ViterbiParser returns only the single most probable parse tree
viterbi_parser = ViterbiParser(grammar)
best_trees = list(viterbi_parser.parse(words))

print(f"Sentence: \"{sentence}\"\n")
print("Most probable parse tree (ViterbiParser):")
for t in best_trees:
    print(t)
    print(f"\nTotal probability: {t.prob():.8f}")


Sentence: "I saw the man with a telescope"

Most probable parse tree (ViterbiParser):
(S
  (NP I)
  (VP
    (V saw)
    (NP
      (NP (Det the) (N man))
      (PP (P with) (NP (Det a) (N telescope)))))) (p=0.0001512)

Total probability: 0.00015120


To justify *why* this parse wins, we need to see the competing interpretation too. `ViterbiParser` only returns the single best parse, so we use `InsideChartParser`, which enumerates every valid parse tree ranked by probability, so we can extract both readings of the sentence.

In [ ]:
from nltk.parse import pchart

# InsideChartParser enumerates ALL valid parse trees, ranked by probability,
# which lets us see both interpretations of the sentence (not just the winner).
inside_parser = pchart.InsideChartParser(grammar)
all_trees = sorted(list(inside_parser.parse(words)), key=lambda t: t.prob(), reverse=True)

print(f"Total number of valid parses found: {len(all_trees)}\n")
for idx, t in enumerate(all_trees, start=1):
    print(f"--- Parse {idx} (probability = {t.prob():.8f}) ---")
    print(t)
    print()


Total number of valid parses found: 2

--- Parse 1 (probability = 0.00015120) ---
(S
  (NP I)
  (VP
    (V saw)
    (NP
      (NP (Det the) (N man))
      (PP (P with) (NP (Det a) (N telescope)))))) (p=0.0001512)

--- Parse 2 (probability = 0.00010800) ---
(S
  (NP I)
  (VP
    (V saw)
    (VP_PP
      (NP (Det the) (N man))
      (PP (P with) (NP (Det a) (N telescope)))))) (p=0.000108)



In [ ]:
def rule_breakdown(tree, grammar):
    """
    Walks a parse tree and returns the list of (rule string, probability)
    for every production actually used to build that tree — read directly
    off the tree, nothing hard-coded.
    """
    rules = []

    def walk(t):
        if isinstance(t, nltk.Tree):
            lhs = t.label()
            rhs = tuple(child.label() if isinstance(child, nltk.Tree) else child for child in t)
            # find the matching PCFG production so we can read off its probability
            for prod in grammar.productions(lhs=nltk.grammar.Nonterminal(lhs)):
                prod_rhs = tuple(str(s) for s in prod.rhs())
                if prod_rhs == rhs:
                    rules.append((f"{lhs} -> {' '.join(rhs)}", prod.prob()))
                    break
            for child in t:
                walk(child)

    walk(tree)
    return rules


def show_parse_math(tree, grammar, label):
    print(f"{label} (tree probability = {tree.prob():.8f})")
    print(tree)
    print()
    running_product = 1.0
    for rule_str, prob in rule_breakdown(tree, grammar):
        print(f"  {rule_str:28s} [{prob}]")
        running_product *= prob
    print(f"\n  Product of rule probabilities = {running_product:.8f}")
    print(f"  Matches tree.prob()?           {abs(running_product - tree.prob()) < 1e-12}\n")
    return running_product


best_tree = all_trees[0]
competing_tree = all_trees[1]

print("=" * 70)
best_prob = show_parse_math(best_tree, grammar, "SELECTED PARSE (ViterbiParser's choice)")
print("=" * 70)
competing_prob = show_parse_math(competing_tree, grammar, "COMPETING PARSE")
print("=" * 70)

ratio = best_prob / competing_prob
print(f"\nSelected parse probability : {best_prob:.8f}")
print(f"Competing parse probability: {competing_prob:.8f}")
print(f"The selected parse is {ratio:.3f}x more probable than the alternative.")


SELECTED PARSE (ViterbiParser's choice) (tree probability = 0.00015120)
(S
  (NP I)
  (VP
    (V saw)
    (NP
      (NP (Det the) (N man))
      (PP (P with) (NP (Det a) (N telescope)))))) (p=0.0001512)

  S -> NP VP                   [1.0]
  NP -> I                      [0.1]
  VP -> V NP                   [0.7]
  V -> saw                     [1.0]
  NP -> NP PP                  [0.6]
  NP -> Det N                  [0.3]
  Det -> the                   [0.8]
  N -> man                     [0.5]
  PP -> P NP                   [1.0]
  P -> with                    [1.0]
  NP -> Det N                  [0.3]
  Det -> a                     [0.2]
  N -> telescope               [0.5]

  Product of rule probabilities = 0.00015120
  Matches tree.prob()?           True

COMPETING PARSE (tree probability = 0.00010800)
(S
  (NP I)
  (VP
    (V saw)
    (VP_PP
      (NP (Det the) (N man))
      (PP (P with) (NP (Det a) (N telescope)))))) (p=0.000108)

  S -> NP VP                   [1.0]
  NP -> I  

### Why ViterbiParser picked this interpretation

Both parses share every rule **except** at the one point in the derivation where the prepositional phrase *with a telescope* gets attached:

- **Selected parse** (p ≈ 0.0001512) attaches the PP *low*, inside the object noun phrase, via `NP -> NP PP [0.6]`. This is the reading **"the man [who has] a telescope"** — the man possesses the telescope.
- **Competing parse** (p ≈ 0.0001080) attaches the PP *high*, to the verb phrase, via `VP -> V VP_PP [0.3]` followed by `VP_PP -> NP PP [1.0]`. This is the reading **"saw ... with a telescope"** — I used the telescope as an instrument to see him.

Every other rule in the two trees is identical (`S -> NP VP`, `NP -> 'I'`, `V -> 'saw'`, `NP -> Det N` used twice, both `Det -> ...`, both `N -> ...`, `PP -> P NP`, `P -> 'with'`), so those factors cancel out of the comparison. The whole decision reduces to comparing just the differing rule(s) at the attachment point:

```
Selected (low attachment):   NP -> NP PP        = 0.6
Competing (high attachment): VP -> V VP_PP [0.3] × VP_PP -> NP PP [1.0] = 0.3
```

Since **0.6 > 0.3**, the low-attachment tree has the larger product of rule probabilities once every rule in the derivation is multiplied together (`P(tree) = ∏ P(rule)`), and `ViterbiParser` — which always returns the tree that maximizes this product — selects it. This is exactly the PCFG disambiguation principle: syntactic ambiguity is resolved purely by comparing the total probability mass of each complete derivation, with no external world knowledge involved.

### Analysis requirement

Do not hard-code the final probability.

Use the parse tree returned by the parser to determine which rules were selected. Then calculate the probability of the competing interpretation from the corresponding rule probabilities.

Your explanation should make clear that **ViterbiParser selects the parse with the highest product of rule probabilities**.

# Part 2: Advanced Dependency Parsing Using spaCy

Use the **spaCy NLP library** to perform dependency parsing and graph traversal on the following sentence:

> **The exhausted researchers at the institute discovered a new vaccine that completely prevents the viral mutation.**


## 1. Initialize and Parse

Load the `en_core_web_sm` model in spaCy and process the sentence.

Return the parsed object and store it as **`parsed_doc`**.

In [ ]:
import spacy

nlp = spacy.load("en_core_web_sm")

sentence2 = "The exhausted researchers at the institute discovered a new vaccine that completely prevents the viral mutation."
parsed_doc = nlp(sentence2)

# Quick sanity check of the parse
for token in parsed_doc:
    print(f"{token.text:<15}{token.pos_:<8}{token.head.text:<15}{token.dep_}")


The            DET     researchers    det
exhausted      ADJ     researchers    amod
researchers    NOUN    discovered     nsubj
at             ADP     researchers    prep
the            DET     institute      det
institute      NOUN    at             pobj
discovered     VERB    discovered     ROOT
a              DET     vaccine        det
new            ADJ     vaccine        amod
vaccine        NOUN    discovered     dobj
that           PRON    prevents       nsubj
completely     ADV     prevents       advmod
prevents       VERB    vaccine        relcl
the            DET     mutation       det
viral          ADJ     mutation       amod
mutation       NOUN    prevents       dobj
.              PUNCT   discovered     punct


## 2. Isolate the Root

Create a function that accepts **`parsed_doc` as its only parameter**.

Iterate through the document to find the structural center of the sentence, where the dependency tag is `"ROOT"`.

Return this token and save it as **`root_node`**.


In [ ]:
def find_root(parsed_doc):
    """Returns the token whose dependency relation is ROOT."""
    for token in parsed_doc:
        if token.dep_ == "ROOT":
            return token
    return None


root_node = find_root(parsed_doc)
print("Root node:", root_node.text)


Root node: discovered


## 3. Extract the Graph Terminals

Write a function that requires **both `root_node` and `parsed_doc`** as inputs.

1. Check the `.children` of `root_node` to find the nominal subject (`nsubj` dependency), isolating **`researchers`** as `start_node`.
2. Find the token whose text is **`mutation`** in `parsed_doc`, isolating it as `end_node`.
3. Return both nodes.

In [ ]:
def find_start_end_nodes(root_node, parsed_doc):
    """
    Finds:
      - start_node: the child of root_node with dependency relation nsubj
      - end_node: the token whose text is "mutation"
    """
    start_node = None
    for child in root_node.children:
        if child.dep_ == "nsubj":
            start_node = child
            break

    end_node = None
    for token in parsed_doc:
        if token.text == "mutation":
            end_node = token
            break

    return start_node, end_node


start_node, end_node = find_start_end_nodes(root_node, parsed_doc)
print("Start node:", start_node.text)
print("End node:  ", end_node.text)


Start node: researchers
End node:   mutation


## 4. Graph Traversal — Shortest Path

Treat the dependency parse as an **undirected graph**. From any token, you may move to:

- its `.head`, or
- any of its `.children`.

Write a **Breadth-First Search (BFS)** function that takes `start_node` and `end_node` as inputs.

Find the shortest path between the two words and return it as a list of tokens named **`dependency_path`**.

In [ ]:
from collections import deque

def bfs_dependency_path(start_node, end_node):
    """
    Treats the dependency parse as an undirected graph (edges to .head and
    to each of .children) and finds the shortest path between start_node
    and end_node using BFS. Returns the path as a list of tokens.
    """
    if start_node.i == end_node.i:
        return [start_node]

    visited = {start_node.i}
    queue = deque([[start_node]])

    while queue:
        path = queue.popleft()
        current = path[-1]

        # neighbors = its head, plus all of its children
        neighbors = list(current.children) + [current.head]

        for neighbor in neighbors:
            if neighbor.i == current.i or neighbor.i in visited:
                continue
            new_path = path + [neighbor]
            if neighbor.i == end_node.i:
                return new_path
            visited.add(neighbor.i)
            queue.append(new_path)

    return []  # no path found


dependency_path = bfs_dependency_path(start_node, end_node)
print("Shortest dependency path:", " -> ".join(t.text for t in dependency_path))


Shortest dependency path: researchers -> discovered -> vaccine -> prevents -> mutation


## 5. Path Analysis

Write a function that accepts **`dependency_path`** as its input.

Iterate through **this list of tokens only**, not the whole sentence, and print their syntactic details in the following format:

| Word | POS | Head | Dependency |
|---|---|---|---|
| researchers | NOUN | discovered | nsubj |
| ... | ... | ... | ... |

This should extract only the most important syntactic skeleton connecting the two concepts.

In [ ]:
def print_path_analysis(dependency_path):
    """Prints POS/head/dependency info for only the tokens on dependency_path."""
    print(f"{'Word':<15}{'POS':<10}{'Head':<15}{'Dependency':<10}")
    print("-" * 50)
    for token in dependency_path:
        print(f"{token.text:<15}{token.pos_:<10}{token.head.text:<15}{token.dep_:<10}")


print_path_analysis(dependency_path)


Word           POS       Head           Dependency
--------------------------------------------------
researchers    NOUN      discovered     nsubj     
discovered     VERB      discovered     ROOT      
vaccine        NOUN      discovered     dobj      
prevents       VERB      vaccine        relcl     
mutation       NOUN      prevents       dobj      


## 6. Visual Verification

Pass the original **`parsed_doc`** into spaCy's `displacy` module to render the full dependency tree.

Use the resulting diagram to visually verify that the table generated in Step 5 matches the shortest path in the dependency tree.

In [ ]:
from spacy import displacy

# Renders the full dependency tree inline (in Jupyter/Colab this displays as an SVG)
displacy.render(parsed_doc, style="dep", jupyter=True, options={"distance": 100})
